# Guichet: retrieval (step 5)
Runtime > Change runtime type > **T4 GPU**. Run cells one at a time and check each output.

In [ ]:
# 1. Mount Drive, get the latest repo, rebuild the chunks
from google.colab import drive
drive.mount('/content/drive')
!test -d /content/guichet-rag || git clone -q https://github.com/AfoDyDigitals/guichet-rag.git /content/guichet-rag
%cd /content/guichet-rag
!git pull -q
!python data/make_chunks.py
!mkdir -p /content/drive/MyDrive/guichet-rag

In [ ]:
# 2. Install libraries (1-2 minutes)
!pip -q install rank_bm25 sentence-transformers

In [ ]:
# 3. Load the models. Check: prints 'cuda'
import json, sys, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
sys.path.insert(0, 'retrieval')
from search import Retriever, load_or_embed, METHODS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
embedder = SentenceTransformer('intfloat/multilingual-e5-small', device=device)
reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', max_length=512, device=device)

In [ ]:
# 4. Embed all chunks (2-5 minutes the first time, seconds after: saved to Drive)
CACHE = '/content/drive/MyDrive/guichet-rag'
retrievers = {}
for name in ['section', 'fixed', 'sentence']:
    chunks = [json.loads(l) for l in open(f'data/chunks_{name}.jsonl')]
    emb = load_or_embed(embedder, chunks, f'{CACHE}/emb_{name}.npy')
    retrievers[name] = Retriever(chunks, embedder, emb, reranker)
    print(name, emb.shape)

In [ ]:
# 5. Smoke test: one of your questions through all 4 methods
q = json.loads(open('evaluation/questions.jsonl').readline())
r = retrievers['section']
print('Q:', q['question'], '| answer:', q['section_id'])
for m in METHODS:
    top = [r.chunks[i] for i in r.search(m, q['question'], 10)]
    hit = any(q['section_id'] in c['section_ids'] for c in top)
    print(f"\n{m:7} answer in top 10: {hit}")
    for c in top[:3]:
        print('   ', c['chunk_id'], c['text'][:90])

## Step 6: score everything

In [ ]:
# 6. Score all question sets x 3 chunkings x 4 methods (5-10 minutes)
import pandas as pd
sys.path.insert(0, 'evaluation')
from evaluate import evaluate, load

question_sets = {'human_en': load('evaluation/questions.jsonl'),
                 'human_fr': load('evaluation/questions_fr.jsonl'),
                 'synthetic': load('evaluation/questions_synthetic.jsonl')}
print({k: len(v) for k, v in question_sets.items()})
rows, ranks = evaluate(retrievers, question_sets, METHODS)
results = pd.DataFrame(rows)
results.to_csv(f'{CACHE}/results.csv', index=False)
with open(f'{CACHE}/ranks.jsonl', 'w') as f:
    f.writelines(json.dumps(x, ensure_ascii=False) + '\n' for x in ranks)
print('saved to Drive')

In [ ]:
# 7. The headline table: recall@5, exact passage found
for qs in question_sets:
    t = results.query('level == "section" and questions == @qs').pivot(index='method', columns='chunking', values='recall@5')
    print(f'\n{qs}  (recall@5, exact passage)')
    print(t.loc[METHODS, ['section', 'fixed', 'sentence']].to_string())